In [18]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [19]:
from src.config import DEFAULT_STGCN_PARAMS_V1, POSE_DATASET_ROOT, DATASET_ROOT
from src.rwf2000 import RWF2000PoseDataset, RWF2000Dataset
import skelbumentations as S
import torch

train_dataset = RWF2000PoseDataset(POSE_DATASET_ROOT, split="train", max_people=4)


In [22]:
def augment_pose(tensor):
    """
    tensor shape: [3, T, V, M]
    channels: x, y, confidence
    """
    opposite_coco_points = [
        [5, 6],  # left shoulder, right shoulder
        [7, 8],  # left elbow, right elbow
        [9, 10], # left wrist, right wrist
        [11, 12], # left hip, right hip
        [13, 14], # left knee, right knee
        [15, 16], # left ankle, right ankle
    ]

    left_arm = [5, 7, 9]
    right_arm = [6, 8, 10]
    left_leg = [11, 13, 15]
    right_leg = [12, 14, 16]


    pipeline = S.Compose([
        # Frame occlusion
        S.SelectRandomFrames(
            [S.WholeOcclusion()], 
            min_num=25, 
            max_num=50, 
            p=1.0, 
        ), 
        # Interpolate one randomly selected body part
        S.SelectRandomWithBorder(
            [
                S.OneOf([
                    S.SpecificOcclusion(left_arm),
                    S.SpecificOcclusion(right_arm),
                    S.SpecificOcclusion(left_leg),
                    S.SpecificOcclusion(right_leg),
                ])
            ],
            [S.InterpolateOcclusions()],
            min_num=3,
            max_num=10,
            p=1.0,
        ),

        # Interpolate whole skeleton
        S.SelectRandomWithBorder(
            [S.WholeOcclusion()],
            [S.InterpolateOcclusions()],
            min_num=3,
            max_num=10,
            p=1.0,
        ),

        # Swap random joints
        S.SelectRandomFrames(
            [S.SwapPerturbation()],
            min_num=1,
            max_num=10,
            contiguous=False,
            p=1.0,
        ),

        # interpolate randomly selected bodypart 
        S.SelectRandomFrames(
            [S.MirrorPerturbation(opposite_coco_points)], 
            min_num=1, 
            max_num=4, 
            p=1.0,
        )
    ])

    tensor = tensor.clone()
    C, T, V, M = tensor.shape

    for person_index in range(M):
        person_pose = tensor[:, :, :, person_index] # [3, T, V]

        if person_pose[2].sum() == 0: # continue if no person exists
            continue

        # pose sequence has to be numpy array with format (T, V, C)
        keypoints = person_pose.permute(1, 2, 0).cpu().numpy().copy()

        # Skelbumentations expects a mask indicating joints that are already missing
        # confidence 0 means keypoint was already missing 
        invalid = (person_pose[2] == 0).cpu().numpy().copy()
        augmented = pipeline(keypoints=keypoints, invalid=invalid)

        augmented_person_pose = torch.from_numpy(augmented["keypoints"]).permute(2, 0, 1)
        tensor[:, :, :, person_index] = augmented_person_pose

    return tensor


In [23]:
tensor, label = train_dataset[40]

augmented_tensor = augment_pose(tensor)

print("Original shape: ", tensor.shape)
print("Augmented shape:", augmented_tensor.shape)

print("Tensor changed:",
      not torch.equal(tensor, augmented_tensor))

difference = tensor != augmented_tensor

changed_frames = difference.any(dim=0).any(dim=1)

print("Changed frames:")
print(torch.where(changed_frames)[0].tolist())

Original shape:  torch.Size([3, 150, 17, 4])
Augmented shape: torch.Size([3, 150, 17, 4])
Tensor changed: True
Changed frames:
[1, 2, 3, 4, 5, 6, 49, 56, 66, 85, 94, 95, 96, 96, 97, 97, 98, 99, 100, 101, 102, 103, 104, 105, 105, 106, 106, 107, 107, 108, 108, 109, 109, 110, 110, 111, 111, 112, 112, 113, 113, 114, 114, 115, 115, 116, 116, 117, 117, 118, 119, 120, 121, 122, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 132, 133, 133, 134, 134, 135, 135, 136, 136, 137, 137, 138, 138, 139, 140, 141, 142, 143]


In [24]:
for person in range(tensor.shape[-1]):
    person_changed = not torch.equal(
        tensor[:, :, :, person],
        augmented_tensor[:, :, :, person],
    )

    print(f"Person {person}: {person_changed}")

Person 0: True
Person 1: True
Person 2: False
Person 3: False


In [25]:
print(
    augmented_tensor[:2].min().item(),
    augmented_tensor[:2].max().item(),
)

0.0 1.0
